# Smart HVAC Energy Analytics & Predictive Maintenance System

## Phase 4 — Feature Engineering and Dataset Preparation

This notebook prepares the processed HVAC simulation dataset for fault detection model training.

The workflow includes:

- Removing irrelevant and empty columns
- Encoding the health status target
- Creating balanced training data
- Separating features and target
- Splitting the data into training and testing sets
- Saving the final datasets for model development

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

## 1. Load Processed Dataset

The preprocessed HVAC simulation dataset generated during the previous stage is loaded as the starting point for feature engineering.

In [2]:
processed_simulation = pd.read_csv(
    "../outputs/processed_simulation.csv"
)

print(processed_simulation.shape)

processed_simulation.head()

(1871233, 41)


,Datetime,RTU_COMP_WATT,RTU_OA_FLOW,RTU_OA_HUM,RTU_OA_TEMP,RTU_RA_FLOW,RTU_RA_HUM,RTU_RA_TEMP,RTU_REFG_COND_PRES,RTU_REFG_COND_TEMP,...,RTU_REFG_COND_TEMP_1,RTU_REFG_COND_TEMP_2,RTU_REFG_DISC_PRES_1,RTU_REFG_DISC_PRES_2,RTU_REFG_DISC_TEMP_1,RTU_REFG_DISC_TEMP_2,RTU_REFG_SUCT_PRES_1,RTU_REFG_SUCT_PRES_2,RTU_REFG_SUCT_TEMP_1,RTU_REFG_SUCT_TEMP_2
0,2018-07-20 01:00:00,2097.6797,181.93083,49.361565,55.040030,2941.1106,50.000000,75.200000,24763394.0,79.043396,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2018-07-20 01:01:00,2089.0981,181.93083,59.133976,55.033990,2941.1106,59.898830,73.353580,24861452.0,79.312790,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2018-07-20 01:02:00,2089.8665,181.93083,59.105840,55.028000,2941.1106,59.870330,73.096176,24846182.0,79.270930,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2018-07-20 01:03:00,2091.4324,181.93083,59.077980,55.022015,2941.1106,59.842117,72.839096,24822794.0,79.206710,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2018-07-20 01:04:00,2093.2980,181.93083,59.050636,55.016030,2941.1106,59.814423,72.583115,24796212.0,79.133650,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Dataset Overview

The dataset structure and dimensions are inspected before feature engineering to confirm the available observations and sensor variables.

In [3]:
print("="*60)
print("FEATURE ENGINEERING")
print("="*60)

print(f"Rows    : {processed_simulation.shape[0]:,}")
print(f"Columns : {processed_simulation.shape[1]}")

processed_simulation.info()

FEATURE ENGINEERING
Rows    : 1,871,233
Columns : 41
<class 'pandas.DataFrame'>
RangeIndex: 1871233 entries, 0 to 1871232
Data columns (total 41 columns):
 #   Column                Dtype  
---  ------                -----  
 0   Datetime              str    
 1   RTU_COMP_WATT         float64
 2   RTU_OA_FLOW           float64
 3   RTU_OA_HUM            float64
 4   RTU_OA_TEMP           float64
 5   RTU_RA_FLOW           float64
 6   RTU_RA_HUM            float64
 7   RTU_RA_TEMP           float64
 8   RTU_REFG_COND_PRES    float64
 9   RTU_REFG_COND_TEMP    float64
 10  RTU_REFG_DISC_PRES    float64
 11  RTU_REFG_DISC_TEMP    float64
 12  RTU_REFG_SUCT_PRES    float64
 13  RTU_REFG_SUCT_TEMP    float64
 14  RTU_SA_FAN_WATT       float64
 15  RTU_SA_FLOW           float64
 16  RTU_SA_HUM            float64
 17  RTU_SA_TEMP           float64
 18  RTU_SEN_CAPA          float64
 19  RTU_STG_STA           float64
 20  RTU_TOT_CAPA          float64
 21  RTU_TOT_WATT          float64
 22  

## 3. Remove Completely Empty Features

Columns containing only missing values provide no information for model training and are removed from the working dataset.

In [5]:
# Remove columns that contain only NaN values
processed_simulation = processed_simulation.dropna(axis=1, how="all")

print("Shape after removing empty columns:")
print(processed_simulation.shape)

Shape after removing empty columns:
(1871233, 24)


## 4. Remove Non-Model Features

Columns that are not required for the binary fault detection task are removed.

- `Datetime` is a timestamp rather than a sensor feature.
- `fault_type` and `severity` are not used because the current model predicts binary health status.
- `ZA_TEMP_SPT` is excluded from the current feature set.

The remaining columns represent the sensor features used for fault detection.

In [6]:
columns_to_drop = [
    "Datetime",
    "fault_type",
    "severity",
    "ZA_TEMP_SPT"
]

processed_simulation.drop(
    columns=columns_to_drop,
    inplace=True,
    errors="ignore"
)

processed_simulation.head()

,RTU_COMP_WATT,RTU_OA_FLOW,RTU_OA_HUM,RTU_OA_TEMP,RTU_RA_FLOW,RTU_RA_HUM,RTU_RA_TEMP,RTU_REFG_COND_PRES,RTU_REFG_COND_TEMP,RTU_REFG_DISC_PRES,...,RTU_SA_FLOW,RTU_SA_HUM,RTU_SA_TEMP,RTU_SEN_CAPA,RTU_STG_STA,RTU_TOT_CAPA,RTU_TOT_WATT,ZA_HUM,ZA_TEMP,health_status
0,2097.6797,181.93083,49.361565,55.040030,2941.1106,50.000000,75.200000,24763394.0,79.043396,25093532.0,...,2941.1106,98.70536,52.272797,14631.9375,1.0,17422.904,2192.1084,59.926773,73.611046,Healthy
1,2089.0981,181.93083,59.133976,55.033990,2941.1106,59.898830,73.353580,24861452.0,79.312790,25198948.0,...,2941.1106,100.00000,53.288155,12823.3100,1.0,17655.139,2183.5269,59.898830,73.353580,Healthy
2,2089.8665,181.93083,59.105840,55.028000,2941.1106,59.870330,73.096176,24846182.0,79.270930,25183480.0,...,2941.1106,100.00000,53.024975,12819.8160,1.0,17625.334,2184.2952,59.870330,73.096176,Healthy
3,2091.4324,181.93083,59.077980,55.022015,2941.1106,59.842117,72.839096,24822794.0,79.206710,25158652.0,...,2941.1106,100.00000,52.775420,12811.6670,1.0,17576.230,2185.8610,59.842117,72.839096,Healthy
4,2093.2980,181.93083,59.050636,55.016030,2941.1106,59.814423,72.583115,24796212.0,79.133650,25130132.0,...,2941.1106,100.00000,52.532455,12801.4100,1.0,17519.484,2187.7268,59.814423,72.583115,Healthy


## 5. Verify Final Feature Set

The remaining columns and dataset dimensions are checked to confirm the feature set before target encoding and dataset balancing.

In [7]:
print(processed_simulation.columns.tolist())
print(processed_simulation.shape)

['RTU_COMP_WATT', 'RTU_OA_FLOW', 'RTU_OA_HUM', 'RTU_OA_TEMP', 'RTU_RA_FLOW', 'RTU_RA_HUM', 'RTU_RA_TEMP', 'RTU_REFG_COND_PRES', 'RTU_REFG_COND_TEMP', 'RTU_REFG_DISC_PRES', 'RTU_REFG_DISC_TEMP', 'RTU_REFG_SUCT_PRES', 'RTU_REFG_SUCT_TEMP', 'RTU_SA_FAN_WATT', 'RTU_SA_FLOW', 'RTU_SA_HUM', 'RTU_SA_TEMP', 'RTU_SEN_CAPA', 'RTU_STG_STA', 'RTU_TOT_CAPA', 'RTU_TOT_WATT', 'ZA_HUM', 'ZA_TEMP', 'health_status']
(1871233, 24)


## 6. Save Cleaned Feature Dataset

The cleaned simulation dataset is saved for use in the subsequent stages of the fault detection pipeline.

In [8]:
processed_simulation.to_csv(
    "../outputs/processed_simulation.csv",
    index=False
)

print("Cleaned processed_simulation.csv saved successfully!")

Cleaned processed_simulation.csv saved successfully!


## 7. Encode Health Status

The categorical health status is converted into a binary target:

- `0` → Healthy
- `1` → Fault

This format is required for binary classification.

In [9]:
processed_simulation["health_status"] = (
    processed_simulation["health_status"]
    .map({
        "Healthy": 0,
        "Fault": 1
    })
)

In [10]:
print(processed_simulation["health_status"].value_counts())

health_status
1    1727292
0     143941
Name: count, dtype: int64


## 8. Separate Features and Target

The dataset is divided into:

- `X` — HVAC sensor features used as model inputs
- `y` — binary health status used as the prediction target

In [12]:
X = processed_simulation.drop(
    columns=["health_status"]
)

y = processed_simulation["health_status"]

print(X.shape)
print(y.shape)

(1871233, 23)
(1871233,)


### Final Model Features

The feature names are displayed to verify the variables that will be supplied to the fault detection model.

In [ ]:
print("Feature Columns\n")
for col in X.columns:
    print(col)

Feature Columns

RTU_COMP_WATT
RTU_OA_FLOW
RTU_OA_HUM
RTU_OA_TEMP
RTU_RA_FLOW
RTU_RA_HUM
RTU_RA_TEMP
RTU_REFG_COND_PRES
RTU_REFG_COND_TEMP
RTU_REFG_DISC_PRES
RTU_REFG_DISC_TEMP
RTU_REFG_SUCT_PRES
RTU_REFG_SUCT_TEMP
RTU_SA_FAN_WATT
RTU_SA_FLOW
RTU_SA_HUM
RTU_SA_TEMP
RTU_SEN_CAPA
RTU_STG_STA
RTU_TOT_CAPA
RTU_TOT_WATT
ZA_HUM
ZA_TEMP


## 9. Analyze Class Distribution

The original distribution of healthy and faulty observations is examined before balancing the dataset.

In [14]:
print(y.value_counts())

print()

print(y.value_counts(normalize=True) * 100)

Series([], Name: count, dtype: int64)

Series([], Name: proportion, dtype: float64)


In [16]:
processed_simulation = pd.read_csv("../outputs/processed_simulation.csv")

## 10. Balance Healthy and Faulty Classes

The fault detection model should not be biased toward the majority class.

To create a balanced training dataset, the number of faulty samples is matched to the number of healthy samples through random sampling.

The resulting dataset is then shuffled to remove ordering effects.

A fixed random seed ensures reproducibility.

In [21]:
healthy_data = processed_simulation[
    processed_simulation["health_status"] == 0
]

fault_data = processed_simulation[
    processed_simulation["health_status"] == 1
]

fault_sample = fault_data.sample(
    n=len(healthy_data),
    random_state=42
)

balanced_data = pd.concat(
    [healthy_data, fault_sample],
    ignore_index=True
)

balanced_data = balanced_data.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

### Verify Class Balance

The class distribution is checked after balancing to confirm that healthy and faulty samples are equally represented.

In [22]:
balanced_data["health_status"].value_counts()

health_status
1    143941
0    143941
Name: count, dtype: int64

In [23]:
balanced_data["health_status"].value_counts(normalize=True)*100

health_status
1    50.0
0    50.0
Name: proportion, dtype: float64

## 11. Create Final Feature and Target Sets

The balanced dataset is separated into model inputs (`X`) and the binary target (`y`) for training and evaluation.

In [24]:
X = balanced_data.drop(columns=["health_status"])

y = balanced_data["health_status"]

In [25]:
print(X.shape)
print(y.shape)

(287882, 23)
(287882,)


## 12. Train-Test Split

The balanced dataset is divided into training and testing subsets.

- 80% of the data is used for training.
- 20% is reserved for testing.
- Stratification preserves the healthy/fault class ratio in both subsets.
- A fixed random state ensures reproducibility.

In [26]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [27]:
y_train.value_counts()

health_status
1    115153
0    115152
Name: count, dtype: int64

In [28]:
y_test.value_counts()

health_status
0    28789
1    28788
Name: count, dtype: int64

## 13. Save Training and Testing Datasets

The final training and testing datasets are saved in both Pickle and CSV formats.

These files are used by subsequent model training and evaluation stages.

In [29]:
X_train.to_pickle("../outputs/X_train.pkl")
X_test.to_pickle("../outputs/X_test.pkl")

y_train.to_pickle("../outputs/y_train.pkl")
y_test.to_pickle("../outputs/y_test.pkl")

X_train.to_csv("../outputs/X_train.csv", index=False)
X_test.to_csv("../outputs/X_test.csv", index=False)

y_train.to_csv("../outputs/y_train.csv", index=False)
y_test.to_csv("../outputs/y_test.csv", index=False)